# Model attribution - feature & temporal importance

Loads the trained checkpoints (same plumbing as `lstm1_evaluate_full_onesample.ipynb`) and produces two figures:

1. **Per-attribute occlusion importance** - mask each input in turn (observed attributes: value -> channel mean, presence-mask -> 0; weather forecast: NWP tensor -> its per-feature mean) and measure how much the forecast moves. Model-agnostic, robust. Reported in physical units (°C / % / m/s, and wrap-around degrees for wind direction).
2. **Contribution-over-time (Integrated Gradients)** - per-timestep attribution of the encoder input, aggregated over channels and shown per day-before-forecast. Directly shows how far back the model actually looks.

Set `MODEL_KEY` to the model you want to attribute.


In [ ]:
import json
import math
import time
import numpy as np
import pandas as pd
import os
from random import randint

import torch
import torch.optim as optim
import torch.utils.data as data
import matplotlib.pyplot as plt

import joblib

from utils.reproducibility import seed_everything, tf_func, clip_grad_func
from utils.loss_fn import masked_mae_multi as loss_fn
from utils.create_dataset_v1 import make_input_data, StationDataset
from utils.models import (AttrSeq2SeqLSTM as seq2seq_model, 
                          AttrSeq2SeqLSTM_v1 as seq2seq_model_v1,
                          AttrSeq2SeqCNNLSTM as seq2seq_cnn_model,
                          AttrLSTM as norm_model, 
                          AttrLSTMv1 as norm_model_v1, 
                          AttrSeq2SeqAttnLSTM as seq2seq_attn_model,
                          train_one_epoch, 
                          evaluate,
                          train_one_epoch_scaler,
                          evaluate_scaler,
                          model_run
                          )
from utils.utils import read_forecast_data

from sklearn.model_selection import train_test_split
from lstm1__const__ import TRAIN_FILES, TEST_FILES

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

INP_ATTRS = None
OUT_ATTRS = None

INP_ATTR_SIZE = None
OUT_ATTR_SIZE = None

USE_SCALER = True

print(f'Using device: {DEVICE}')
ENCODER_IN_DIM = 0
DECODER_TF_DIM = 0

SEED = 42
HIDDEN_DIM = 256

FORECAST_DATA = read_forecast_data('./data/forecast_hr.csv', device=DEVICE, use_scaler=USE_SCALER)
print('FORECAST_DATA.shape', FORECAST_DATA.shape)
# FORECAST_DATA: (T, n_days, day_dim) — forecast is grouped per day for per-step alignment
FORECAST_DAYS = FORECAST_DATA.shape[1]   # number of forecast days (e.g. 4)
FORECAST_DIM = FORECAST_DATA.shape[2]    # per-day feature dim (e.g. 10)
CNN_CONFIG = [[64, 128], 1, 1]

def _pick_best_epoch(val_losses, tol=0.0):
    v = np.asarray(val_losses, dtype=float)
    if v.size == 0:
        return None
    m = v.min()
    thresh = m * (1.0 + tol) if m > 0 else m + tol
    return int(np.argmax(v <= thresh))   # earliest epoch within tol (relative) of the min

def loadModel(DAY, loadbest=True):
    MODEL_LOADED = None
    epoch_files = {}
    for fname in os.listdir(f"models/{DAY}/"):
        if 'epoch_' not in fname:
            continue
        epoch_files[int(fname.split('epoch_')[1].split('.pth')[0])] = f"models/{DAY}/{fname}"
    filename = epoch_files[max(epoch_files)] if epoch_files else None
    def _load(path):
        try:
            return torch.load(path, map_location=DEVICE, weights_only=False)
        except TypeError:
            return torch.load(path, map_location=DEVICE)
    if filename is not None:
        MODEL_LOADED = _load(filename)
        # Prefer the best-VALIDATION epoch (earliest at the min) over the last epoch, which for
        # these early-peaking models is the overfit tail. prev_losses is stored in the checkpoint.
        if loadbest:
            val = (MODEL_LOADED.get('prev_losses') or {}).get('test', {}).get('mean')
            if val:
                best = _pick_best_epoch(val)
                if best in epoch_files and epoch_files[best] != filename:
                    filename = epoch_files[best]
                    MODEL_LOADED = _load(filename)
                    print(f'  loadbest: epoch {best} (val {val[best]:.5f}) instead of last {max(epoch_files)}')
        extra = MODEL_LOADED['extra']
        INP_ATTRS = extra['inputs']
        OUT_ATTRS = extra['outputs']
        ENCODER_IN_DIM = extra['encoder_in_dim']
        DECODER_TF_DIM = extra['decoder_tf_dim']
        USE_SCALER = extra['use_scaler']
        LOOKBACK = extra['lookback']
        LOOKFORWARD = extra['lookforward']
        DROPOUT = extra['dropout']
        USE_WAVELET = extra['use_wavelet'] if 'use_wavelet' in extra else USE_WAVELET
        WAVELET_LVL = extra['wavelet_lvl'] if 'wavelet_lvl' in extra else WAVELET_LVL
        STN_MATCH_ATTRIBUTE = extra['stn_match_attr'] if 'stn_match_attr' in extra else STN_MATCH_ATTRIBUTE
        cnn_config = extra['cnn_config'] if 'cnn_config' in extra else CNN_CONFIG
        forecast_dim = extra['forecast_dim'] if 'forecast_dim' in extra else None
        HIDDEN_DIM = extra['hidden_dim'] if 'hidden_dim' in extra else HIDDEN_DIM
        OUT_ATTR_SIZE = len(OUT_ATTRS)
        model = None
        print('model type:', extra['model_type'])
        if 'model_type' in extra and extra['model_type'] == 'seq2seq':
            model = seq2seq_model(ENCODER_IN_DIM, DECODER_TF_DIM, OUT_ATTR_SIZE, hidden_dim=HIDDEN_DIM)
        elif 'model_type' in extra and extra['model_type'] == 'seq2seq_v1':
            model = seq2seq_model_v1(ENCODER_IN_DIM, DECODER_TF_DIM, OUT_ATTR_SIZE, hidden_dim=HIDDEN_DIM, forecast_dim=forecast_dim, forecast_days=FORECAST_DAYS)
        elif 'model_type' in extra and extra['model_type'] == 'seq2seq_attn':
            model = seq2seq_attn_model(ENCODER_IN_DIM, DECODER_TF_DIM, OUT_ATTR_SIZE, hidden_dim=HIDDEN_DIM, forecast_dim=forecast_dim, forecast_days=FORECAST_DAYS)
        elif 'model_type' in extra and extra['model_type'] == 'seq2seq_cnn':
            model = seq2seq_cnn_model(
                ENCODER_IN_DIM, DECODER_TF_DIM, OUT_ATTR_SIZE,
                hidden_dim=HIDDEN_DIM,
                forecast_dim=forecast_dim, forecast_days=FORECAST_DAYS,
                # CNN experiments (toggle here):
                #   current 3-layer pooling : cnn_channels=(64,128,256), pool_size=2, dilation=1
                #   2-layer pooling         : cnn_channels=(64,128),     pool_size=2, dilation=1
                #   dilated, no pooling     : cnn_channels=(64,128),     pool_size=1, dilation=2
                cnn_channels=cnn_config[0], pool_size=cnn_config[1], dilation=cnn_config[2],
            )
        elif 'model_type' in extra and extra['model_type'] == 'plain_lstm':
            model = norm_model(ENCODER_IN_DIM, DECODER_TF_DIM, OUT_ATTR_SIZE, hidden_dim=HIDDEN_DIM, forecast_dim=forecast_dim, forecast_days=FORECAST_DAYS)
        elif 'model_type' in extra and extra['model_type'] == 'v1':
            model = norm_model_v1(ENCODER_IN_DIM, DECODER_TF_DIM, OUT_ATTR_SIZE, hidden_dim=HIDDEN_DIM)
        else:
            model = norm_model(ENCODER_IN_DIM, DECODER_TF_DIM, OUT_ATTR_SIZE, hidden_dim=HIDDEN_DIM)

        # model.load_state_dict(MODEL_LOADED, strict=True)
        state_dict = MODEL_LOADED["model"]
        if any(k.startswith("_orig_mod.") for k in state_dict):
            state_dict = {k[len("_orig_mod."):]: v for k, v in state_dict.items()}
        model.load_state_dict(state_dict, strict=True)
        # _unwrap_model(model).load_state_dict(MODEL_LOADED["model"], strict=True)
        prev_losses = MODEL_LOADED["prev_losses"]

        print('Model loaded:', filename)
    model.to(DEVICE) 
    return {'model': model, 'model_info': MODEL_LOADED}
LOADED = 1
del LOADED

In [ ]:
from lstm1__const__ import DAYS

try:
    print(LOADED)
except Exception:
    MODELS  = {}
    RESULTS = {}
    for i0 in DAYS:
        if i0 not in MODELS:
            MODELS[i0] = {}
            RESULTS[i0] = {}
        for i1 in DAYS[i0]:
            print(i0, i1, DAYS[i0][i1])
            MODELS[i0][i1] = loadModel(DAYS[i0][i1])
            RESULTS[i0][i1] = {}
    LOADED = 'MODEL AND DATA LOADED'


In [ ]:
DATA_DIR = 'model_data_1/'
FILE_SUFFIX = '_hr_avg.csv'


# # LOOKBACK_DAYS = 60
# # INTERVAL = 60 * 60
# # LOOKBACK_TIME = 60 * 60 * 24 * LOOKBACK_DAYS
# # LOOKBACK = round(LOOKBACK_TIME / INTERVAL)
# LOOKBACK = None

# # LOOKFORWARD_TIME = 60 * 60 * 24 * 7
# # LOOKFORWARD = round(LOOKFORWARD_TIME / INTERVAL)
# LOOKFORWARD = None

BATCH_SIZE = 256
BATCH_SIZE_EFF = 256
BATCH_ACCU = int(BATCH_SIZE_EFF / BATCH_SIZE)
BATCH_COUNT = int(BATCH_SIZE_EFF * math.floor(5000 / BATCH_SIZE_EFF))


# INP_ATTRS = None
# OUT_ATTRS = None

# INP_ATTR_SIZE = None
# OUT_ATTR_SIZE = None

# USE_SCALER = False

# print(f'Using device: {DEVICE}')
# ENCODER_IN_DIM = 0
# DECODER_TF_DIM = 0

# HIDDEN_DIM = 256


SEED = 42
STN_MATCH_ATTRIBUTE = 'lat_long_h'

ATTR_MATCH = {
    'temperature pt100': 'Temp',
    'relative humidity': 'RH',
    'wind speed': 'WSpd',
    'sin_wind': 'Wdir',
    'cos_wind': 'Wdir',
    'sin_dir': 'Wdir',
    'sin_dir': 'Wdir',
    'solar radiation': 'Sol',
    'pm1.0': 'pm1',
    'pm2.5': 'pm2p5',
    'pm10': 'pm10'
}

SCALER = {}
INP_ATTRS = ['Temperature PT100', 'Relative Humidity', 'Wind speed', 'sin_dir', 'cos_dir', 'Solar radiation']

for attr in INP_ATTRS:
    p = f"./scalers/{attr}.gz"
    if os.path.exists(p):
        SCALER[attr.lower()] = joblib.load(p)
        print(f'scaler file - OK: ./scalers/{attr}.gz')
    else:
        SCALER[attr.lower()] = None
        print(f'scaler file - not exist: ./scalers/{attr}.gz')

stn_match_file = open(DATA_DIR + 'station_match_1.json')
station_match = json.loads(stn_match_file.read())
stn_match_file.close()


In [ ]:
EVAL_ORDER = ['Temp', 'RH', 'WSpd', 'WDir']


def createDataset(MODEL_DATA, station_idx, dataset_type='train'):
    extra = MODEL_DATA['extra']
    OUT_ATTRS = extra['outputs']
    LOOKBACK = extra['lookback']
    LOOKFORWARD = extra['lookforward']
    USE_WAVELET = extra['use_wavelet'] if 'use_wavelet' in extra else USE_WAVELET
    WAVELET_LVL = extra['wavelet_lvl'] if 'wavelet_lvl' in extra else WAVELET_LVL
    BATCHNORM = extra['batchnorm'] if 'batchnorm' in extra else BATCHNORM
    STN_MATCH_ATTRIBUTE = extra['stn_match_attr'] if 'stn_match_attr' in extra else STN_MATCH_ATTRIBUTE
    # Must match training: if the model was trained without the absolute-year feature, zero it here
    # too. Default True for older checkpoints (trained with year_norm included).
    INCLUDE_YEAR = extra.get('include_year', True)

    DATA_DIR = 'model_data_preprocessed_nosol/' + ATTR_MATCH[OUT_ATTRS[0]] + '_60/'

    if dataset_type == 'train' or dataset_type == 'eval':
        file = TRAIN_FILES[station_idx]
        stn_data = np.load(f'{DATA_DIR}/{file}.npz')
        stn_info = np.array(station_match[file][STN_MATCH_ATTRIBUTE], dtype=np.float32)
        time_feats, target, target_mask, valid_idx = stn_data['time_feats'], stn_data['target'], stn_data['target_mask'], stn_data['valid_idx']
        series = stn_data['series']
        inp = stn_data['input']
        inp_mask = stn_data['input_mask']
        forecast_idx = stn_data['forecast_idx']
    
        # Date-based split on the forecast target period: TEST = windows whose 168h target
        # starts on/after SPLIT_DATE; TRAIN = windows whose target ends before SPLIT_DATE.
        # Windows straddling the boundary are dropped so no training target overlaps the test
        # period. Exact for a Jan-1 cutoff: time_feats[:, 4] is the absolute year (years since
        # 1970, base_year=0), so the year comparison IS the date comparison.
        SPLIT_DATE = np.datetime64('2025-01-01')
        SPLIT_YEAR = int(SPLIT_DATE.astype('datetime64[Y]').astype(int))  # years since 1970; exact for a Jan-1 cutoff
        year_of = time_feats[:, 4]
        tgt_start_year = year_of[valid_idx + LOOKBACK]
        tgt_end_year   = year_of[valid_idx + LOOKBACK + LOOKFORWARD - 1]
        is_test  = tgt_start_year >= SPLIT_YEAR
        is_train = tgt_end_year   <  SPLIT_YEAR
        train_idx, train_forecast = valid_idx[is_train], forecast_idx[is_train]
        test_idx,  test_forecast  = valid_idx[is_test],  forecast_idx[is_test]

        if dataset_type == 'train':
            return StationDataset(
            series, time_feats, target, target_mask, train_idx, stn_info, 
            LOOKBACK, LOOKFORWARD, forecast=FORECAST_DATA, forecast_idx=train_forecast,
            use_wavelet=USE_WAVELET, wavelet_level=WAVELET_LVL, inp=inp, inp_mask=inp_mask)
        if dataset_type == 'eval':
            return StationDataset(
            series, time_feats, target, target_mask, test_idx, stn_info,
            LOOKBACK, LOOKFORWARD, forecast=FORECAST_DATA, forecast_idx=test_forecast,
            use_wavelet=USE_WAVELET, wavelet_level=WAVELET_LVL, inp=inp, inp_mask=inp_mask)
    if dataset_type == 'test':
        file = TEST_FILES[station_idx]
        stn_data = np.load(f'{DATA_DIR}/{file}.npz')
        stn_data = np.load(f'{DATA_DIR}/{file}.npz')
        stn_info = np.array(station_match[file][STN_MATCH_ATTRIBUTE], dtype=np.float32)
        time_feats, target, target_mask, valid_idx = stn_data['time_feats'], stn_data['target'], stn_data['target_mask'], stn_data['valid_idx']
        series = stn_data['series']
        inp = stn_data['input']
        inp_mask = stn_data['input_mask']
        forecast_idx = stn_data['forecast_idx']
        if not INCLUDE_YEAR:
            time_feats[:, 4] = 0.0   # series[:, :5] IS time_feats, so zero both
            series[:, 4] = 0.0
        return StationDataset(
            series, time_feats, target, target_mask, valid_idx, stn_info,
            LOOKBACK, LOOKFORWARD, forecast=FORECAST_DATA, forecast_idx=forecast_idx,
            use_wavelet=USE_WAVELET, wavelet_level=WAVELET_LVL, inp=inp, inp_mask=inp_mask)


def _sample_start_keys(dataset):
    """Map each sample -> a key identifying its decoder start time (first entry of dec_tf).

    dec_tf[0] for sample i is time_feats[valid_idx[i] + lookback], which uniquely
    identifies the window's start time. valid_idx / time_feats do not depend on the
    model (wavelet or not), so a given start-time key maps to the same sample index
    across all models of the same attribute.
    Returns {start_key: sample_idx}.
    """
    starts = np.asarray(dataset.valid_idx) + dataset.lookback
    tf0_all = dataset.time_feats[starts].detach().cpu().numpy().astype(float)   # (N, 5)
    tf0_all = np.round(tf0_all, 6)
    return {tuple(row): i for i, row in enumerate(tf0_all)}


def _fmt_start_time(tf0):
    ms, mc, ds_, dc, yr = [float(v) for v in tf0]
    minute = (math.atan2(ms, mc) / (2 * math.pi)) % 1.0 * 1440
    doy    = (math.atan2(ds_, dc) / (2 * math.pi)) % 1.0 * 365
    year   = int(round(1970 + yr))
    hh, mm = int(minute // 60), int(round(minute % 60))
    return f'{year} doy~{doy:.0f} {hh:02d}:{mm:02d}'


In [ ]:
# Integrated Gradients backend (optional dependency)
try:
    from captum.attr import IntegratedGradients
    HAVE_CAPTUM = True
except Exception:
    HAVE_CAPTUM = False
    print('captum not installed - IG figure will be skipped. Install with:  pip install captum')


In [ ]:
# --- Encoder channel layout (preprocessing: series = [time(5) | mask(K) | value(K)]) ---
TIME_DIM = 5
INP_ATTRS_SERIES = ['temperature pt100', 'relative humidity', 'wind speed', 'sin_dir', 'cos_dir']
K = len(INP_ATTRS_SERIES)
_val = lambda j: TIME_DIM + K + j     # value channel for input attr j
_msk = lambda j: TIME_DIM + j         # presence-mask channel for input attr j

# Group value+mask channels per physical attribute (wind dir = sin+cos together).
# 'Forecast (NWP)' is NOT an encoder channel - it is the separate (B, n_days, day_dim) forecast
# tensor, so it carries 'forecast': True and the occlusion routine mean-fills `fc` instead.
CHANNEL_GROUPS = {
    'Temp':          {'value': [_val(0)], 'mask': [_msk(0)]},
    'RH':            {'value': [_val(1)], 'mask': [_msk(1)]},
    'WSpd':          {'value': [_val(2)], 'mask': [_msk(2)]},
    'WDir':          {'value': [_val(3), _val(4)], 'mask': [_msk(3), _msk(4)]},
    'Time features': {'value': list(range(TIME_DIM)), 'mask': []},
    'Forecast': {'value': [], 'mask': [], 'forecast': True},
}
VALUE_CHANNELS = [_val(j) for j in range(K)]   # physical measurement channels (temp, RH, wspd, sin, cos)
MASK_CHANNELS  = [_msk(j) for j in range(K)]   # presence-mask channels
# require completeness only on the target attribute's mask channel(s) -> far larger eligible pool
ATTR_TO_MASK = {'temp': [_msk(0)], 'RH': [_msk(1)], 'WSpd': [_msk(2)], 'WDir': [_msk(3), _msk(4)]}


# --- Physical units ---------------------------------------------------------------------------
# Occlusion effects are reported in sensor units, not in the model's scaled space, so bars are
# comparable across attributes and readable as "the forecast moves by X degC / % / m/s / degrees".
# Temp / RH / WSpd are single-channel and were scaled at preprocessing -> inverse_transform.
# WDir is 2-channel [sin, cos] and unscaled -> convert to compass degrees and use the wrap-around
# (shortest-arc) angular difference, so 350 deg vs 10 deg is a 20 deg change, not 340.
ATTR_UNITS = {'Temp': '°C', 'RH': '%', 'WSpd': 'm/s', 'WDir': '°'}
ATTR_NAMES = {'Temp': 'Temperature', 'RH': 'Relative Humidity', 'WSpd': 'Wind Speed', 'WDir': 'Wind Direction'}


def get_scaler(model_info):
    """Scaler for this model's output attribute, or None (WDir sin/cos, or model trained unscaled).
    Matches the convention in lstm1_evaluate_full_wday.py: only single-output models are scaled."""
    extra = model_info['extra']
    outs = [a.lower() for a in extra['outputs']]
    if len(outs) != 1 or not extra.get('use_scaler', True):
        return None
    return SCALER.get(outs[0])


def to_physical(pred, attr, scaler):
    """(B, T, C) model output -> (B, T) numpy in physical units.
    WDir: [sin, cos] -> compass degrees in [0, 360). Others: inverse-scaled to the sensor unit."""
    p = pred.detach().cpu().float().numpy().astype(np.float64)
    if attr == 'WDir':
        return np.degrees(np.arctan2(p[:, :, 0], p[:, :, 1])) % 360.0
    p = p.reshape(p.shape[0], -1)
    if scaler is not None:
        p = scaler.inverse_transform(p.reshape(-1, 1)).reshape(p.shape)
    return p


def phys_mean_abs_diff(p, base, attr):
    """Mean |p - base| over every (sample, horizon step) in physical units.
    WDir uses the shortest-arc angular difference instead of a plain subtraction."""
    d = (p - base + 180.0) % 360.0 - 180.0 if attr == 'WDir' else p - base
    return float(np.abs(d).mean())


def _sample_complete(sample, input_channels=None):
    """True if the window has NO missing values (mask == 1 everywhere).
    Checks the target mask and the given encoder input-mask channels (default: all)."""
    enc, dec, m, y, stn, last, fc = sample
    chans = MASK_CHANNELS if input_channels is None else input_channels
    inp_ok = bool((enc[:, chans] == 1).all())    # all input observations present over the whole lookback
    tgt_ok = bool((m == 1).all())                # target fully present over the whole horizon
    return inp_ok and tgt_ok


def build_batch(model_info, dataset_type='test', n_samples=24, seed=0,
                require_complete=True, input_channels=None):
    """Collect a batch of windows across stations into stacked tensors (enc, dec, m, y, stn, last, fc).
    require_complete=True keeps only windows with no missing input/target values, so attribution is
    measured on genuinely complete data (occlusion effects are not confounded by pre-masked gaps)."""
    rng = np.random.default_rng(seed)
    files = TEST_FILES if dataset_type == 'test' else TRAIN_FILES
    per = max(1, n_samples // len(files))
    cols = [[] for _ in range(7)]
    for si in range(len(files)):
        try:
            ds = createDataset(model_info, si, dataset_type)
        except Exception:
            continue
        if ds is None or len(ds) == 0:
            continue
        taken = 0
        for idx in rng.permutation(len(ds)):       # shuffled scan so we can skip incomplete windows
            sample = ds[int(idx)]
            if require_complete and not _sample_complete(sample, input_channels):
                continue
            for c, v in zip(cols, sample):
                c.append(v)
            taken += 1
            if taken >= per or len(cols[0]) >= n_samples:
                break
        if len(cols[0]) >= n_samples:
            break
    if len(cols[0]) == 0:
        raise RuntimeError("no complete windows found - relax require_complete or pass input_channels "
                           "(e.g. only the target attribute value+mask channels).")
    if len(cols[0]) < n_samples:
        print(f"warning: only {len(cols[0])} complete windows collected (< {n_samples} requested)")
    return tuple(torch.stack(c, 0) for c in cols)


print('models available:', list(DAYS.keys()))


In [ ]:
MODEL_KEY = 'LSTM-ED-FC'   # which model in DAYS to attribute (must exist and carry NWP for -FC models)
N_SAMPLES = 128             # windows pooled across test stations
IG_STEPS  = 128             # Integrated-Gradients path steps (higher = more accurate, slower)
print('using MODEL_KEY =', MODEL_KEY)


## Figure 1 - Per-attribute occlusion importance

For each output attribute, mask each input in turn and measure the mean absolute change in the forecast. Higher bar = the model relies on it more.

- **Observed attributes / time features** - masking sets the value channel to its dataset mean **and** the presence-mask channel to 0, so the model treats the attribute as genuinely absent.
- **Forecast (NWP)** - the weather forecast is a separate `(B, n_days, day_dim)` tensor, not an encoder channel, so it is occluded by replacing it with its per-feature mean over the batch and forecast days (same "replace by the mean, keep the shape" idea). The model's has-forecast flag is derived internally from the decoder time features and cannot be switched off from outside, so this measures reliance on the forecast *content*. For models without NWP input (`forecast_dim is None`) the bar is exactly 0 by construction.

Bars are in **physical units**: Temp / RH / WSpd are inverse-scaled with their training scaler (°C, %, m/s) and WDir is converted from the `[sin, cos]` pair to compass degrees, with the change measured as the wrap-around (shortest-arc) angular difference.


In [ ]:
@torch.no_grad()
def occlusion_importance(model, batch, groups, attr, scaler):
    """Mask each input group and measure the mean absolute change of the forecast in PHYSICAL units.

    Encoder groups : value channels -> channel mean, presence-mask channels -> 0.
    Forecast group : the NWP tensor -> its per-feature mean over (batch, forecast day).
    The change is measured per (sample, horizon step) after inverse-scaling; WDir uses the
    wrap-around angular difference in degrees."""
    enc, dec, m, y, stn, last, fc = batch
    run = lambda e, f: to_physical(
        model_run(model, e, dec, last_val=last, station_feats=stn, forecast=f), attr, scaler)
    base = run(enc, fc)
    ch_mean = enc.mean(dim=(0, 1))
    fc_mean = fc.mean(dim=(0, 1))          # (day_dim,) - mean over batch AND forecast days
    out = {}
    for name, g in groups.items():
        e2, f2 = enc, fc
        if g.get('forecast'):
            # f2 = fc_mean.expand_as(fc).contiguous()
            f2 = torch.zeros(fc.shape, dtype=torch.float32, device=DEVICE)
        if g['value'] or g['mask']:
            e2 = enc.clone()
            for c in g['value']:
                e2[:, :, c] = ch_mean[c]
            for c in g['mask']:
                e2[:, :, c] = 0.0
        out[ATTR_NAMES[name] if name in ATTR_NAMES else name] = phys_mean_abs_diff(run(e2, f2), base, attr)
    return out


fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, attr in zip(axes.ravel(), EVAL_ORDER):
    if MODEL_KEY not in DAYS or attr not in DAYS[MODEL_KEY]:
        ax.set_title(f'{attr}: model not available'); ax.axis('off'); continue
    md = MODELS[MODEL_KEY][attr]; model = md['model']; model.eval()
    scaler = get_scaler(md['model_info'])
    has_nwp = getattr(model, 'forecast_dim', None) is not None
    batch = build_batch(md['model_info'], 'test', N_SAMPLES, input_channels=ATTR_TO_MASK.get(attr))
    print(f'{attr}: {batch[0].shape[0]} complete windows, scaler={"yes" if scaler is not None else "no"}, nwp={has_nwp}')
    imp = occlusion_importance(model, batch, CHANNEL_GROUPS, attr, scaler)
    names = list(imp.keys())
    # colors = ['darkorange' if CHANNEL_GROUPS[n].get('forecast') else 'steelblue' for n in names]
    ax.bar(names, [imp[n] for n in names])
    ax.set_title(f'{ATTR_NAMES[attr]} - occlusion importance ({MODEL_KEY})' + ('' if has_nwp else '  [no NWP input]'))
    ax.set_ylabel(f'mean |change| in forecast ({ATTR_UNITS[attr]})')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## Figure 2 - Contribution over history (Integrated Gradients)

Integrated Gradients attributes the whole forecast back to each encoder timestep; we sum |attribution| over channels and pool into days. **Day 0 = most recent.** If contribution concentrates in the first few days, that's empirical support for a shorter lookback window.

> Runtime scales with `N_SAMPLES` x `IG_STEPS` x 168-step rollout. Start small (24 / 24) and raise for smoother curves.


In [ ]:
def ig_over_time(model, batch, n_steps=128, value_only=True):
    """Integrated Gradients over encoder timesteps.
    value_only=True holds the binary mask + cyclic sin/cos time-feature channels at their
    real values (baseline == input there), so IG only interpolates the physical value
    channels. This avoids integrating through out-of-distribution fractional masks /
    off-circle sin-cos points, which otherwise destroys convergence.
    Returns (per-timestep importance (T,), info dict with completeness diagnostics)."""
    enc, dec, m, y, stn, last, fc = batch
    enc = enc.detach()
    def fwd(e):
        p = model_run(model, e, dec, last_val=last, station_feats=stn, forecast=fc)
        return p.reshape(p.size(0), -1).mean(dim=1)   # scalar per sample
    ig = IntegratedGradients(fwd)
    vmean = enc.mean(dim=(0, 1), keepdim=True)
    if value_only:
        base = enc.clone()
        base[:, :, VALUE_CHANNELS] = vmean[:, :, VALUE_CHANNELS]   # only value channels move mean->actual
    else:
        base = vmean.expand_as(enc).contiguous()
    # cuDNN's fused RNN backward only runs in train mode; disable it so the native
    # RNN kernel (whose backward works in eval mode) is used for the attribution pass.
    with torch.enable_grad(), torch.backends.cudnn.flags(enabled=False):
        attr, delta = ig.attribute(enc, baselines=base, n_steps=n_steps,
                                    internal_batch_size=enc.size(0),
                                    return_convergence_delta=True)
    t_imp = attr.abs().sum(dim=2).mean(dim=0).detach().cpu().numpy()   # (T,)
    total = attr.reshape(attr.size(0), -1).sum(dim=1).detach()         # ~ f(input)-f(baseline) per sample
    delta = delta.detach()
    info = {'mean_abs_delta': delta.abs().mean().item(),
            'mean_abs_total': total.abs().mean().item()}
    info['rel_error'] = info['mean_abs_delta'] / (info['mean_abs_total'] + 1e-12)
    return t_imp, info


def per_day_ago(t_imp):
    rev = t_imp[::-1]                       # index 0 = most recent hour (closest to forecast)
    nd = len(rev) // 24
    return rev[:nd * 24].reshape(nd, 24).sum(axis=1)   # index 0 = most recent day


# if not HAVE_CAPTUM:
#     print('captum not installed - skipping IG figure. Install with:  pip install captum')
# else:
#     fig, axes = plt.subplots(2, 2, figsize=(14, 9))
#     for ax, attr in zip(axes.ravel(), EVAL_ORDER):
#         if MODEL_KEY not in DAYS or attr not in DAYS[MODEL_KEY]:
#             ax.set_title(f'{attr}: model not available'); ax.axis('off'); continue
#         md = MODELS[MODEL_KEY][attr]; model = md['model']; model.eval()
#         batch = build_batch(md['model_info'], 'test', N_SAMPLES, seed=1, input_channels=ATTR_TO_MASK.get(attr))
#         print(f'{attr}: {batch[0].shape[0]} complete windows')
#         t_imp, info = ig_over_time(model, batch, n_steps=IG_STEPS)
#         day_imp = per_day_ago(t_imp)
#         print(f"{attr:5s}  mean|delta|={info['mean_abs_delta']:.3g}  "
#               f"mean|out-base|={info['mean_abs_total']:.3g}  rel_error={info['rel_error']:.1%}")
#         ax.plot(np.arange(len(day_imp)), day_imp, marker='o', ms=3)
#         ax.set_title(f"{attr} - IG over history ({MODEL_KEY})\n" +
#                      f"rel. convergence error {info['rel_error']:.0%}")
#         ax.set_xlabel('days before forecast (0 = most recent)')
#         ax.set_ylabel('summed |attribution|')
#         ax.grid(True, alpha=0.3)
#     plt.tight_layout(); plt.show()


## Figure 2b - Contribution over history (occlusion, robust)

Since IG will not converge on this autoregressive model, use occlusion instead: mask each day-block of history (value -> channel mean, presence-mask -> 0; calendar/time features left intact) and measure how much the forecast moves. No path integral, nothing to converge - this is the reliable temporal-importance figure. **Day 0 = most recent.**


In [ ]:
@torch.no_grad()
def occlusion_over_time(model, batch, attr, scaler, block=24):
    """Mask each day-block of history (value -> mean, presence-mask -> 0; time feats kept) and
    measure mean |change| in the forecast, in physical units (same metric as Figure 1, so the two
    figures are directly comparable). No convergence issues. Index 0 = most recent day."""
    enc, dec, m, y, stn, last, fc = batch
    run = lambda e: to_physical(
        model_run(model, e, dec, last_val=last, station_feats=stn, forecast=fc), attr, scaler)
    base = run(enc)
    T = enc.size(1)
    ch_mean = enc.mean(dim=(0, 1))
    nblocks = T // block
    imp = np.zeros(nblocks)
    for b in range(nblocks):
        e2 = enc.clone()
        sl = slice(T - (b + 1) * block, T - b * block)   # b=0 -> most recent day
        for c in VALUE_CHANNELS:
            e2[:, sl, c] = ch_mean[c]
        for c in MASK_CHANNELS:
            e2[:, sl, c] = 0.0
        imp[b] = phys_mean_abs_diff(run(e2), base, attr)
    return imp


fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, attr in zip(axes.ravel(), EVAL_ORDER):
    if MODEL_KEY not in DAYS or attr not in DAYS[MODEL_KEY]:
        ax.set_title(f'{attr}: model not available'); ax.axis('off'); continue
    md = MODELS[MODEL_KEY][attr]; model = md['model']; model.eval()
    scaler = get_scaler(md['model_info'])
    batch = build_batch(md['model_info'], 'test', N_SAMPLES, seed=1, input_channels=ATTR_TO_MASK.get(attr))
    print(f'{attr}: {batch[0].shape[0]} complete windows')
    imp = occlusion_over_time(model, batch, attr, scaler, block=24)
    ax.plot(np.arange(len(imp)), imp, marker='o', ms=3, color='seagreen')
    ax.set_title(f'{ATTR_NAMES[attr]} - occlusion over history ({MODEL_KEY})')
    ax.set_xlabel('days before forecast (0 = most recent)')
    ax.set_ylabel(f'mean |change| when day masked ({ATTR_UNITS[attr]})')
    ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## Notes

- **Occlusion vs IG**: occlusion is model-agnostic and robust but coarse; IG is finer-grained and axiom-satisfying but sensitive to the baseline (here the per-channel mean). Agreement between the two is the strongest evidence.
- **Attention heatmap** (optional third view): `AttrSeq2SeqAttnLSTM` computes softmax attention weights internally but does not return them. To plot a `(horizon x history)` heatmap, modify its `forward` to stash the per-step weights, then re-run. Attention weights are a *proxy* for importance, not a substitute for the two figures above.
- **Autoregressive caveat**: attribution flows through the 168-step rollout, so input importance is entangled across horizons. Use `horizon=` targeting (mean over the whole forecast by default) if you want to see how the dependence shifts with lead time.
